# 08 — Monte-Carlo: fill the data many times, measure accuracy

Simulates the production use of the model: apply it to many validation samples in three ways, measure how the accuracy holds up.

1. **Continuous-burst reconstruction** — pick one validation file (~200 contiguous timesteps from one burst), inpaint every frame, and plot the recovered plasma-density time series against the ground truth.
2. **Per-sample error distribution** — histogram of |Δn|/n across the val set: median, 75th and 95th percentiles, U-Net vs. baseline.
3. **Mask-position robustness** — slide the wedge around the (θ, φ) plane and rerun **pure inference** (no retraining). Tells us whether the model has learned something general about inpainting, or only memorised the trained mask position.

In [ ]:
import os, sys, warnings, numpy as np, matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=DeprecationWarning); warnings.filterwarnings('ignore', category=FutureWarning)
ROOT = os.path.dirname(os.path.abspath('.')) if os.path.basename(os.getcwd())=='notebooks' else os.path.abspath('.')
sys.path.insert(0, os.path.join(ROOT, 'src')); sys.path.insert(0, os.path.join(ROOT, 'MMS-FPI-Data-Gaps'))
import data_pipeline as dp
from model import build_unet3d
from baseline import energy_shell_mean_fill
from skymaps.skymap import Skymap
OUT = os.path.join(ROOT, 'outputs', 'unet_full_3rounds')

In [ ]:
mask_default = dp.synthetic_wedge_mask()
files = dp.find_dist_files(os.path.join(ROOT, 'MMS-FPI-Data-Gaps'))
_, val_files = dp.split_files(files, val_fraction=0.2, seed=0)
Xs, Ys, file_index = [], [], []
for fi, f in enumerate(val_files):
    fb = dp.read_dist_file(f, subsample=24, with_phi=True)
    X, Y = dp.build_inputs(fb, mask_default, use_pitch_angle=True, use_logb=True, temporal_window=1)
    Xs.append(X); Ys.append(Y); file_index.extend([fi] * X.shape[0])
X = np.concatenate(Xs, 0); Y = np.concatenate(Ys, 0); file_index = np.array(file_index)
m = build_unet3d(base_filters=10, input_shape=(32, 16, 32, 5))
m.load_weights(os.path.join(OUT, 'model_latest.h5'))
P = m.predict(X, verbose=0)
y_cube = Y[..., 0]; p_cube = P[..., 0]
b_cube = energy_shell_mean_fill(dp.apply_mask(y_cube, mask_default), mask_default)
unet_in = y_cube.copy(); unet_in[:, mask_default] = p_cube[:, mask_default]
base_in = y_cube.copy(); base_in[:, mask_default] = b_cube[:, mask_default]

## 1. Continuous-burst reconstruction


In [ ]:
def densities(cube, n):
    sk = Skymap(n, name='x'); sk.skymap = np.zeros((n, 32, 16, 32), dtype=np.float64)
    sk.skymap[:] = dp.to_physical_space(cube); return np.asarray(sk.momsTS.density)
burst_idx = np.where(file_index == 0)[0]
d_true = densities(y_cube[burst_idx],  burst_idx.size)
d_unet = densities(unet_in[burst_idx], burst_idx.size)
d_base = densities(base_in[burst_idx], burst_idx.size)
plt.figure(figsize=(11, 4))
plt.plot(d_true, 'k-', lw=2.0, label='true'); plt.plot(d_unet, 'C0-', lw=1.4, label='U-Net inpaint'); plt.plot(d_base, 'C3--', lw=1.2, label='baseline inpaint')
plt.xlabel('burst timestep'); plt.ylabel('density (Skymap moments)'); plt.title('Continuous-burst reconstruction (file 0)'); plt.legend(); plt.show()

## 2. Per-sample error histogram


In [ ]:
N = min(200, y_cube.shape[0])
sel = np.sort(np.random.default_rng(11).choice(y_cube.shape[0], N, replace=False))
dT = densities(y_cube[sel], N); dU = densities(unet_in[sel], N); dB = densities(base_in[sel], N)
ok = np.abs(dT) > 1e-30; eU = np.abs(dT[ok]-dU[ok])/np.abs(dT[ok]); eB = np.abs(dT[ok]-dB[ok])/np.abs(dT[ok])
for n, e in (('U-Net', eU), ('baseline', eB)): print(f'{n:8s}  median {np.median(e):.3f}  75th {np.percentile(e,75):.3f}  95th {np.percentile(e,95):.3f}')
plt.figure(figsize=(8,4)); bins=np.linspace(0,1,41)
plt.hist(np.clip(eB,0,1), bins=bins, color='C3', alpha=0.6, label=f'baseline (median {np.median(eB):.2f})')
plt.hist(np.clip(eU,0,1), bins=bins, color='C0', alpha=0.7, label=f'U-Net (median {np.median(eU):.2f})')
plt.xlabel('|Δn|/n'); plt.ylabel('# samples'); plt.legend(); plt.title('Density-error distribution'); plt.show()

## 3. Mask-position robustness (no retraining — pure inference under shifted masks)


In [ ]:
def make_wedge(t0, p0, ts=8, ps=16):
    msk = np.zeros((32,16,32), bool)
    for t in range(ts):
        for p in range(ps): msk[:, (t0+t)%16, (p0+p)%32] = True
    return msk
configs = [('default (trained)', make_wedge(0,0)), ('shift phi +8', make_wedge(0,8)), ('shift phi +16', make_wedge(0,16)),
           ('shift theta +4', make_wedge(4,0)), ('shift theta +8', make_wedge(8,0)), ('shift both', make_wedge(4,16))]
subset = np.random.default_rng(0).choice(y_cube.shape[0], 200, replace=False)
results = []
for name, msk in configs:
    Xn = np.zeros((subset.size, 32, 16, 32, 5), np.float32)
    for k, d in enumerate([-1, 0, 1]):
        src = np.clip(subset + d, 0, y_cube.shape[0]-1)
        Xn[..., k] = dp.apply_mask(y_cube[src], msk)
    Xn[..., 3:] = X[subset][..., 3:]
    Pn = m.predict(Xn, verbose=0)[..., 0]
    Bn = energy_shell_mean_fill(dp.apply_mask(y_cube[subset], msk), msk)
    mse_u = float(((Pn - y_cube[subset])[:, msk]**2).mean()); mse_b = float(((Bn - y_cube[subset])[:, msk]**2).mean())
    results.append((name, mse_u, mse_b)); print(f'{name:22s} U-Net {mse_u:.4e}  baseline {mse_b:.4e}')
labels=[r[0] for r in results]; uu=[r[1] for r in results]; bb=[r[2] for r in results]
ind=np.arange(len(labels)); w=0.38
plt.figure(figsize=(9,4)); plt.bar(ind-w/2, uu, w, color='C0', label='U-Net'); plt.bar(ind+w/2, bb, w, color='C3', label='baseline')
plt.yscale('log'); plt.xticks(ind, labels, rotation=20, ha='right'); plt.ylabel('masked-region MSE'); plt.legend(); plt.title('Mask-position robustness'); plt.show()